# MIMIC-IV-Note PostgreSQL Build Pipeline

Builds the `mimiciv_note` schema using `mimic-code/mimic-iv-note/buildmimic/postgres/`.  
Assumes the PostgreSQL container is already running (managed by `build_mimic.ipynb`), but checks and starts it if needed.  
Each step checks whether it has already been completed and skips it on subsequent runs.

All configuration is read from the same `.env` file as `build_mimic.ipynb`.

| Step | Description |
|------|-------------|
| 1 | Verify Docker is running |
| 2 | Ensure Docker network exists |
| 3 | Build PostgreSQL image (`docker/Dockerfile.db`) |
| 4 | Start PostgreSQL container |
| 5 | Wait for PostgreSQL to be ready |
| 6 | Create `mimiciv_note` schema and tables (`create.sql`) |
| 7 | Load note data (`load_gz.sql`) |
| 8 | Validate note row counts (`validate.sql`) |

> **Run all steps**: *Kernel → Restart & Run All*  
> **Run one step**: call the individual `stepN_*()` function in a new cell.

## Imports & Logging

In [1]:
import json
import logging
import os
import re
import subprocess
import tempfile
import time
from datetime import datetime
from pathlib import Path

import psycopg
from dotenv import load_dotenv

LOG_FORMAT = "%(asctime)s [%(levelname)-8s] %(message)s"
LOG_DATE   = "%Y-%m-%d %H:%M:%S"

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE,
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("build_mimic_notes.log", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

## Configuration

In [2]:
load_dotenv()

POSTGRES_DB        = os.environ.get("POSTGRES_DB",        "mimiciv")
POSTGRES_USER      = os.environ.get("POSTGRES_USER",      "mimicuser")
POSTGRES_PASSWORD  = os.environ.get("POSTGRES_PASSWORD",  "mimicpass")
POSTGRES_PORT      = int(os.environ.get("POSTGRES_PORT",  "5432"))
DB_CONTAINER_NAME  = os.environ.get("DB_CONTAINER_NAME",  "mimic_postgres")
DB_IMAGE_NAME      = os.environ.get("DB_IMAGE_NAME",      "mimic-db")
DOCKER_NETWORK     = os.environ.get("DOCKER_NETWORK",     "mimic-net")
MIMIC_CODE_DIR     = os.environ.get("MIMIC_CODE_DIR",     "./mimic-code")
MIMIC_DATA_DIR     = os.environ.get("MIMIC_DATA_DIR",     "")
MIMIC_NOTE_DATA_DIR = os.environ.get("MIMIC_NOTE_DATA_DIR", "")
LOAD_LIMITS_FILE   = os.environ.get("LOAD_LIMITS_FILE",   "")

MIMIC_NOTE_BUILD_DIR = Path(MIMIC_CODE_DIR) / "mimic-iv-note" / "buildmimic" / "postgres"

def _rel_path_cfg(p) -> str:
    try:
        return str(Path(p).relative_to(Path.cwd()))
    except ValueError:
        return Path(p).name

def _data_dir_cfg(p) -> str:
    s = str(p)
    return f"...{s[-10:]}" if len(s) > 10 else s

# Resolve note data directory: prefer MIMIC_DATA_DIR/note, fall back to MIMIC_NOTE_DATA_DIR
_mimic_data_note_subdir = Path(MIMIC_DATA_DIR) / "note" if MIMIC_DATA_DIR else None
if _mimic_data_note_subdir and _mimic_data_note_subdir.is_dir():
    _note_dir_source = f"MIMIC_DATA_DIR/note ({_data_dir_cfg(_mimic_data_note_subdir)})"
elif MIMIC_NOTE_DATA_DIR:
    _note_dir_source = f"MIMIC_NOTE_DATA_DIR ({_data_dir_cfg(MIMIC_NOTE_DATA_DIR)})"
else:
    _note_dir_source = "(not set)"

print(f"Database          : {POSTGRES_DB}")
print(f"User              : {POSTGRES_USER}")
print(f"Container         : {DB_CONTAINER_NAME}")
print(f"Network           : {DOCKER_NETWORK}")
print(f"Port              : {POSTGRES_PORT}")
print(f"MIMIC_DATA_DIR    : {_data_dir_cfg(MIMIC_DATA_DIR) if MIMIC_DATA_DIR else '(not set)'}")
print(f"Note data dir     : {_note_dir_source}")
print(f"Limits file       : {_rel_path_cfg(LOAD_LIMITS_FILE) if LOAD_LIMITS_FILE else '(not set — no row limits)'}")
print(f"mimic-code        : {_rel_path_cfg(MIMIC_CODE_DIR)}")
print(f"Note build dir    : {_rel_path_cfg(MIMIC_NOTE_BUILD_DIR)}")

Database          : mimiciv
User              : mimicuser
Container         : mimic_postgres
Network           : mimic-net
Port              : 5432
MIMIC_DATA_DIR    : ...files/data
Note data dir     : MIMIC_DATA_DIR/note (.../data/note)
Limits file       : limits.json
mimic-code        : mimic-code
Note build dir    : postgres


## Helper Functions

In [3]:
def _rel_path(p) -> str:
    """Return a path relative to cwd, falling back to the filename only."""
    try:
        return str(Path(p).relative_to(Path.cwd()))
    except ValueError:
        return Path(p).name


def _data_dir_str(p) -> str:
    """For large data directories show '...' + last 10 chars only."""
    s = str(p)
    return f"...{s[-10:]}" if len(s) > 10 else s


def run(cmd: list[str], *, check: bool = True, capture: bool = False) -> subprocess.CompletedProcess:
    """Run a subprocess command with logging. Returns CompletedProcess."""
    log.debug("Executing: %s", " ".join(str(c) for c in cmd))
    try:
        return subprocess.run(cmd, check=check, capture_output=capture, text=True)
    except subprocess.CalledProcessError as e:
        if e.stdout:
            log.error("stdout:\n%s", e.stdout)
        if e.stderr:
            log.error("stderr:\n%s", e.stderr)
        raise


def db_connect() -> psycopg.Connection:
    """Open a psycopg connection from the host to the running PostgreSQL container."""
    return psycopg.connect(
        host="localhost",
        port=POSTGRES_PORT,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        dbname=POSTGRES_DB,
    )


def run_psql_in_docker(
    script_path: Path,
    extra_psql_vars: dict[str, str] | None = None,
    extra_volumes: list[str] | None = None,
    capture: bool = False,
) -> subprocess.CompletedProcess:
    """
    Run a psql script inside a temporary postgres:17 container on DOCKER_NETWORK,
    connecting to DB_CONTAINER_NAME by container name.
    """
    abs_script = Path(script_path).resolve()
    conn_str = (
        f"host={DB_CONTAINER_NAME} "
        f"dbname={POSTGRES_DB} "
        f"user={POSTGRES_USER} "
        f"password={POSTGRES_PASSWORD}"
    )

    cmd = [
        "docker", "run", "--rm",
        "--network", DOCKER_NETWORK,
        "-v", f"{abs_script}:/script.sql:ro",
    ]
    if extra_volumes:
        for vol in extra_volumes:
            cmd += ["-v", vol]

    cmd += ["postgres:17", "psql", conn_str, "-v", "ON_ERROR_STOP=1"]
    if extra_psql_vars:
        for key, val in extra_psql_vars.items():
            cmd += ["-v", f"{key}={val}"]

    cmd += ["-f", "/script.sql"]
    return run(cmd, capture=capture)

## Row Limits & Timed Load Utilities

In [4]:
def load_limits() -> dict | None:
    """
    Load per-table row limits from LOAD_LIMITS_FILE.
    Returns the parsed dict, or None if LOAD_LIMITS_FILE is unset/missing.
    Expected JSON shape: { "default": 10000, "overrides": { "discharge": 50000 } }
    """
    if not LOAD_LIMITS_FILE:
        return None
    path = Path(LOAD_LIMITS_FILE)
    if not path.exists():
        log.warning("LOAD_LIMITS_FILE '%s' not found — loading without row limits.", _rel_path(path))
        return None
    with path.open() as f:
        return json.load(f)


def _row_limit(table: str, limits: dict) -> int | None:
    """Return the row limit for a table, or None for unlimited."""
    override = limits.get("overrides", {}).get(table)
    if override is not None:
        return override
    return limits.get("default")


# MIMIC-IV-Note uses schema-qualified names: \COPY mimiciv_note.discharge FROM PROGRAM ...
_COPY_RE = re.compile(
    r"(\\COPY\s+\w+\.(\w+)\s+FROM\s+PROGRAM\s+')(gzip -dc [^']+)('.*)",
    re.IGNORECASE,
)

_TIMING_RE = re.compile(r"\[TABLE_(START|END)\] (\w+) (\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})")

_TS_FMT = "YYYY-MM-DD HH24:MI:SS"


def _truncate_note_tables() -> None:
    """Truncate all tables in the mimiciv_note schema before a reload."""
    with db_connect() as conn:
        cur = conn.cursor()
        cur.execute(
            """
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'mimiciv_note'
              AND table_type = 'BASE TABLE'
            ORDER BY table_name
            """,
        )
        tables = cur.fetchall()
        if not tables:
            return
        for (table,) in tables:
            cur.execute(f"TRUNCATE TABLE mimiciv_note.{table}")
        conn.commit()
        log.info("Truncated %d note table(s) for clean reload.", len(tables))


def generate_timed_note_load_sql(
    original_sql: Path,
    out_dir: Path,
    limits: dict | None = None,
) -> Path:
    """
    Rewrite load_gz.sql to wrap each \\COPY with SELECT clock_timestamp() statements
    that emit [TABLE_START] / [TABLE_END] markers for per-table load timing.
    """
    lines = []
    for line in original_sql.read_text().splitlines(keepends=True):
        m = _COPY_RE.match(line.rstrip("\n"))
        if m:
            prefix, table, program, suffix = m.groups()
            if limits is not None:
                limit = _row_limit(table, limits)
                if limit is not None:
                    program = f"{program} | head -n {limit + 1}"
                    log.info("  %-30s → limit %d rows", table, limit)
            lines.append(
                f"SELECT '[TABLE_START] {table} ' || to_char(clock_timestamp(), '{_TS_FMT}') AS ts;\n"
            )
            lines.append(f"{prefix}{program}{suffix}\n")
            lines.append(
                f"SELECT '[TABLE_END] {table} ' || to_char(clock_timestamp(), '{_TS_FMT}') AS ts;\n"
            )
        else:
            lines.append(line)

    out_path = out_dir / "load_note_timed.sql"
    out_path.write_text("".join(lines))
    return out_path


def _log_table_timings(stdout: str) -> None:
    """Parse [TABLE_START]/[TABLE_END] markers from psql output and log per-table durations."""
    starts: dict[str, datetime] = {}
    for line in stdout.splitlines():
        m = _TIMING_RE.search(line)
        if not m:
            continue
        kind, table, ts = m.groups()
        try:
            dt = datetime.fromisoformat(ts)
        except ValueError:
            continue
        if kind == "START":
            starts[table] = dt
            log.info("  %-30s  started  %s", table, ts)
        elif kind == "END" and table in starts:
            secs = (dt - starts[table]).total_seconds()
            log.info("  %-30s  finished %s  (%.1f s)", table, ts, secs)

## Steps

In [5]:
def step1_check_docker() -> None:
    log.info("=" * 60)
    log.info("STEP 1: Verifying Docker is running")
    log.info("=" * 60)

    result = run(["docker", "info"], check=False, capture=True)
    if result.returncode != 0:
        raise RuntimeError("Docker is not running or not accessible. Start Docker Desktop and retry.")

    version_result = run(["docker", "version", "--format", "{{.Server.Version}}"],
                         check=False, capture=True)
    version = version_result.stdout.strip() if version_result.returncode == 0 else "unknown"
    log.info("Docker is running. Server version: %s", version)

In [6]:
def step2_ensure_network() -> None:
    log.info("=" * 60)
    log.info("STEP 2: Ensuring Docker network '%s' exists", DOCKER_NETWORK)
    log.info("=" * 60)

    result = run(["docker", "network", "inspect", DOCKER_NETWORK], check=False, capture=True)
    if result.returncode == 0:
        log.info("SKIP: Network '%s' already exists.", DOCKER_NETWORK)
        return

    log.info("Network '%s' not found — creating it ...", DOCKER_NETWORK)
    run(["docker", "network", "create", DOCKER_NETWORK])
    log.info("Network '%s' created successfully.", DOCKER_NETWORK)

In [7]:
def step3_build_image() -> None:
    log.info("=" * 60)
    log.info("STEP 3: Building PostgreSQL image '%s'", DB_IMAGE_NAME)
    log.info("=" * 60)

    result = run(["docker", "image", "inspect", DB_IMAGE_NAME], check=False, capture=True)
    if result.returncode == 0:
        log.info("SKIP: Image '%s' already exists — skipping build.", DB_IMAGE_NAME)
        return

    dockerfile = Path("docker/Dockerfile.db")
    if not dockerfile.exists():
        raise RuntimeError(f"Dockerfile not found at '{dockerfile}'. Run from the project root.")

    log.info("Building image '%s' from %s ...", DB_IMAGE_NAME, dockerfile)
    run(["docker", "build", "-f", str(dockerfile), "-t", DB_IMAGE_NAME, "."])
    log.info("Image '%s' built successfully.", DB_IMAGE_NAME)

In [8]:
def _ensure_container_on_network() -> None:
    """Connect DB_CONTAINER_NAME to DOCKER_NETWORK if it isn't already."""
    inspect = run(
        ["docker", "container", "inspect", DB_CONTAINER_NAME],
        check=False, capture=True,
    )
    if inspect.returncode != 0:
        return
    info = json.loads(inspect.stdout)
    networks = info[0].get("NetworkSettings", {}).get("Networks", {})
    if DOCKER_NETWORK not in networks:
        log.warning(
            "Container '%s' is not on network '%s' — connecting it now ...",
            DB_CONTAINER_NAME, DOCKER_NETWORK,
        )
        run(["docker", "network", "connect", DOCKER_NETWORK, DB_CONTAINER_NAME])
        log.info("Container '%s' connected to network '%s'.", DB_CONTAINER_NAME, DOCKER_NETWORK)
    else:
        log.info("Container '%s' is already on network '%s'.", DB_CONTAINER_NAME, DOCKER_NETWORK)


def step4_start_db() -> None:
    log.info("=" * 60)
    log.info("STEP 4: Starting PostgreSQL container '%s'", DB_CONTAINER_NAME)
    log.info("=" * 60)

    inspect = run(
        ["docker", "container", "inspect", DB_CONTAINER_NAME],
        check=False, capture=True,
    )

    if inspect.returncode == 0:
        info  = json.loads(inspect.stdout)
        state = info[0]["State"]["Status"]

        if state == "running":
            log.info("Container '%s' is already running.", DB_CONTAINER_NAME)
        else:
            log.info("Container '%s' exists but is in state '%s' — starting it ...",
                     DB_CONTAINER_NAME, state)
            run(["docker", "start", DB_CONTAINER_NAME])
            log.info("Container '%s' started.", DB_CONTAINER_NAME)

        _ensure_container_on_network()
        return

    data_dir = Path("data/db").resolve()
    data_dir.mkdir(parents=True, exist_ok=True)
    log.info("Data directory: %s", _data_dir_str(data_dir))

    log.info("Creating and starting container '%s' ...", DB_CONTAINER_NAME)
    run([
        "docker", "run", "-d",
        "--name",      DB_CONTAINER_NAME,
        "--network",   DOCKER_NETWORK,
        "--shm-size",  "256m",
        "-p",          f"{POSTGRES_PORT}:5432",
        "-v",          f"{data_dir}:/var/lib/postgresql/data",
        DB_IMAGE_NAME,
    ])
    log.info("Container '%s' created and started.", DB_CONTAINER_NAME)

In [9]:
def step5_wait_for_db(max_attempts: int = 30, delay: float = 5.0) -> None:
    log.info("=" * 60)
    log.info("STEP 5: Waiting for PostgreSQL to accept connections")
    log.info("=" * 60)
    log.info("Polling every %.0f s (up to %d attempts) ...", delay, max_attempts)

    for attempt in range(1, max_attempts + 1):
        try:
            conn = db_connect()
            conn.close()
            log.info("PostgreSQL is ready (attempt %d/%d).", attempt, max_attempts)
            return
        except Exception as exc:
            log.info(
                "Not ready yet [%d/%d]: %s — retrying in %.0f s ...",
                attempt, max_attempts, exc, delay,
            )
            time.sleep(delay)

    raise RuntimeError(
        f"PostgreSQL did not become ready after {max_attempts} attempts. "
        f"Check logs with: docker logs {DB_CONTAINER_NAME}"
    )

In [10]:
def step6_create_note_schema() -> None:
    log.info("=" * 60)
    log.info("STEP 6: Creating mimiciv_note schema and tables (create.sql)")
    log.info("=" * 60)

    with db_connect() as conn:
        cur = conn.cursor()
        cur.execute("""
            SELECT COUNT(*)
            FROM information_schema.schemata
            WHERE schema_name = 'mimiciv_note'
        """)
        schema_exists = cur.fetchone()[0]

    if schema_exists:
        with db_connect() as conn:
            cur = conn.cursor()
            cur.execute("""
                SELECT COUNT(*)
                FROM information_schema.tables
                WHERE table_schema = 'mimiciv_note'
            """)
            table_count = cur.fetchone()[0]

        if table_count > 0:
            log.info(
                "SKIP: mimiciv_note schema already exists with %d tables.",
                table_count,
            )
            return
        log.info("mimiciv_note schema exists but has no tables — re-running create.sql ...")
    else:
        log.info("mimiciv_note schema not found — running create.sql ...")

    script = MIMIC_NOTE_BUILD_DIR / "create.sql"
    if not script.exists():
        raise RuntimeError(
            f"Script not found: {_rel_path(script)}. "
            f"Ensure mimic-code is cloned at MIMIC_CODE_DIR='{_rel_path(MIMIC_CODE_DIR)}'."
        )

    log.warning("NOTE: create.sql drops and recreates mimiciv_note. Existing note data will be lost.")
    run_psql_in_docker(script)
    log.info("mimiciv_note schema and tables created successfully.")

In [11]:
def _resolve_note_data_path() -> Path:
    """
    Resolve the note data directory using this priority:
      1. MIMIC_DATA_DIR/note  — if MIMIC_DATA_DIR is set and the note/ subdir exists
      2. MIMIC_NOTE_DATA_DIR  — explicit override
    Raises RuntimeError if neither resolves to an existing directory.
    """
    if MIMIC_DATA_DIR:
        candidate = Path(MIMIC_DATA_DIR).resolve() / "note"
        if candidate.is_dir():
            log.info("Note data path resolved from MIMIC_DATA_DIR/note: %s", _data_dir_str(candidate))
            return candidate
        log.info(
            "MIMIC_DATA_DIR/note ('%s') not found — falling back to MIMIC_NOTE_DATA_DIR.",
            _data_dir_str(candidate),
        )

    if MIMIC_NOTE_DATA_DIR:
        path = Path(MIMIC_NOTE_DATA_DIR).resolve()
        if path.is_dir():
            log.info("Note data path resolved from MIMIC_NOTE_DATA_DIR: %s", _data_dir_str(path))
            return path
        raise RuntimeError(
            f"MIMIC_NOTE_DATA_DIR '{_data_dir_str(path)}' does not exist."
        )

    raise RuntimeError(
        "Could not locate note data directory. Set one of:\n"
        "  MIMIC_DATA_DIR  to a path whose note/ subdirectory contains the note CSV.gz files, or\n"
        "  MIMIC_NOTE_DATA_DIR  to the directory containing the note CSV.gz files directly."
    )


def step7_load_note_data() -> None:
    log.info("=" * 60)
    log.info("STEP 7: Loading MIMIC-IV-Note data (load_gz.sql)")
    log.info("=" * 60)
    log.info("NOTE: This step can take a long time for the full dataset (radiology table has ~2.3M rows).")

    note_data_path = _resolve_note_data_path()

    # Verify required CSV.gz files are present
    required_files = ["discharge.csv.gz", "radiology.csv.gz",
                      "discharge_detail.csv.gz", "radiology_detail.csv.gz"]
    missing = [f for f in required_files if not (note_data_path / f).exists()]
    if missing:
        raise RuntimeError(
            f"Missing note CSV files in {_data_dir_str(note_data_path)}: {missing}"
        )

    try:
        with db_connect() as conn:
            cur = conn.cursor()
            cur.execute("SELECT COUNT(*) FROM mimiciv_note.discharge")
            row_count = cur.fetchone()[0]
    except Exception:
        row_count = 0

    if row_count > 0:
        log.info(
            "SKIP: mimiciv_note.discharge already contains %d rows — note data already loaded.",
            row_count,
        )
        return

    log.info("No data found in mimiciv_note.discharge — starting note data load ...")
    _truncate_note_tables()
    log.info("Note data directory: %s", _data_dir_str(note_data_path))
    log.info("This may take a while. Per-table timing will appear below when complete.")

    script = MIMIC_NOTE_BUILD_DIR / "load_gz.sql"
    if not script.exists():
        raise RuntimeError(f"Script not found: {_rel_path(script)}")

    limits = load_limits()
    if limits is not None:
        log.info(
            "Row limits active (from '%s'): default=%s",
            _rel_path(LOAD_LIMITS_FILE), limits.get("default"),
        )

    with tempfile.TemporaryDirectory() as tmp:
        timed_script = generate_timed_note_load_sql(script, Path(tmp), limits=limits)
        result = run_psql_in_docker(
            timed_script,
            extra_psql_vars={"mimic_data_dir": "/data/note"},
            extra_volumes=[f"{note_data_path}:/data/note:ro"],
            capture=True,
        )

    _log_table_timings(result.stdout)
    log.info("Note data loaded successfully.")

In [12]:
def step8_validate_note() -> None:
    log.info("=" * 60)
    log.info("STEP 8: Validating MIMIC-IV-Note row counts (validate.sql)")
    log.info("=" * 60)

    script = MIMIC_NOTE_BUILD_DIR / "validate.sql"
    if not script.exists():
        raise RuntimeError(f"Script not found: {_rel_path(script)}")

    abs_script = script.resolve()
    conn_str = (
        f"host={DB_CONTAINER_NAME} "
        f"dbname={POSTGRES_DB} "
        f"user={POSTGRES_USER} "
        f"password={POSTGRES_PASSWORD}"
    )

    result = run(
        [
            "docker", "run", "--rm",
            "--network", DOCKER_NETWORK,
            "-v", f"{abs_script}:/script.sql:ro",
            "postgres:17",
            "psql", conn_str,
            "-f", "/script.sql",
        ],
        check=False,
        capture=True,
    )

    print(result.stdout)
    if result.stderr.strip():
        log.warning("Validation stderr:\n%s", result.stderr)

    if result.returncode != 0:
        log.warning(
            "validate.sql exited with code %d — some row counts may not match expected values.",
            result.returncode,
        )
    else:
        log.info("Validation completed — all row counts match expected values.")

## Run Pipeline

In [13]:
step1_check_docker()
step2_ensure_network()
step3_build_image()
step4_start_db()
step5_wait_for_db()
step6_create_note_schema()
step7_load_note_data()
step8_validate_note()

print("\n" + "=" * 60)
print("Pipeline complete. mimiciv_note schema is ready.")
print(f"Connect via: psql -h localhost -p {POSTGRES_PORT} -U {POSTGRES_USER} -d {POSTGRES_DB}")
print("Tables: mimiciv_note.discharge, mimiciv_note.radiology,")
print("        mimiciv_note.discharge_detail, mimiciv_note.radiology_detail")
print("=" * 60)

2026-03-21 19:13:30 [INFO    ] ============================================================
2026-03-21 19:13:30 [INFO    ] STEP 1: Verifying Docker is running
2026-03-21 19:13:30 [INFO    ] ============================================================
2026-03-21 19:13:30 [INFO    ] Docker is running. Server version: 29.2.1
2026-03-21 19:13:30 [INFO    ] ============================================================
2026-03-21 19:13:30 [INFO    ] STEP 2: Ensuring Docker network 'mimic-net' exists
2026-03-21 19:13:30 [INFO    ] ============================================================
2026-03-21 19:13:30 [INFO    ] SKIP: Network 'mimic-net' already exists.
2026-03-21 19:13:30 [INFO    ] ============================================================
2026-03-21 19:13:30 [INFO    ] STEP 3: Building PostgreSQL image 'mimic-db'
2026-03-21 19:13:30 [INFO    ] ============================================================
2026-03-21 19:13:30 [INFO    ] SKIP: Image 'mimic-db' already exists — skippi

DROP SCHEMA
CREATE SCHEMA
DROP TABLE
CREATE TABLE
DROP TABLE
CREATE TABLE
DROP TABLE
CREATE TABLE
DROP TABLE
CREATE TABLE


2026-03-21 19:15:25 [INFO    ]   discharge                       started  2026-03-21 23:13:30
2026-03-21 19:15:25 [INFO    ]   discharge                       finished 2026-03-21 23:14:49  (79.0 s)
2026-03-21 19:15:25 [INFO    ]   radiology                       started  2026-03-21 23:14:49
2026-03-21 19:15:25 [INFO    ]   radiology                       finished 2026-03-21 23:15:21  (32.0 s)
2026-03-21 19:15:25 [INFO    ]   discharge_detail                started  2026-03-21 23:15:21
2026-03-21 19:15:25 [INFO    ]   discharge_detail                finished 2026-03-21 23:15:21  (0.0 s)
2026-03-21 19:15:25 [INFO    ]   radiology_detail                started  2026-03-21 23:15:21
2026-03-21 19:15:25 [INFO    ]   radiology_detail                finished 2026-03-21 23:15:25  (4.0 s)
2026-03-21 19:15:25 [INFO    ] Note data loaded successfully.
2026-03-21 19:15:25 [INFO    ] ============================================================
2026-03-21 19:15:25 [INFO    ] STEP 8: Validating MIMIC-

       tbl        | expected_count | observed_count | row_count_check 
------------------+----------------+----------------+-----------------
 discharge        |         331793 |         331793 | PASSED
 discharge_detail |         186138 |         186138 | PASSED
 radiology        |        2321355 |        2321355 | PASSED
 radiology_detail |        6046121 |        6046121 | PASSED
(4 rows)



Pipeline complete. mimiciv_note schema is ready.
Connect via: psql -h localhost -p 5432 -U mimicuser -d mimiciv
Tables: mimiciv_note.discharge, mimiciv_note.radiology,
        mimiciv_note.discharge_detail, mimiciv_note.radiology_detail
